<a href="https://colab.research.google.com/github/Kashish415/LLM_sycophancy_dpo/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers==5.17.0 trl==1.12.0 peft==0.21.0 bitsandbytes==0.50.2 datasets==5.0.1 accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 16.6 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
login()

In [1]:
from getpass import getpass
token = getpass("GitHub token: ")

GitHub token: ··········


In [ ]:
!git clone https://{token}@github.com/Kashish415/LLM_sycophancy_dpo.git

Cloning into 'LLM_sycophancy_dpo'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 30 (delta 6), reused 28 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 7.46 KiB | 7.46 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [ ]:
%cd LLM_sycophancy_dpo
!mkdir -p results

/content/LLM_sycophancy_dpo


In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset("kashishgupta/anti-sycophancy-dpo-cleaned")

README.md:   0%|          | 0.00/449 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.13MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  131kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2545 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/283 [00:00<?, ? examples/s]

In [ ]:
def to_conversational(example):
    return {
        "prompt": [{"role": "user", "content": example["user_input"]}],
        "chosen": [{"role": "assistant", "content": example["chosen"]}],
        "rejected": [{"role": "assistant", "content": example["rejected"]}]
    }

In [ ]:
train_dataset = dataset["train"].map(to_conversational, remove_columns=dataset["train"].column_names)
eval_dataset = dataset["test"].map(to_conversational, remove_columns=dataset["test"].column_names)

Map:   0%|          | 0/2545 [00:00<?, ? examples/s]

Map:   0%|          | 0/283 [00:00<?, ? examples/s]

In [ ]:
from src.model import load_model

model, tokenizer = load_model()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [ ]:
model = prepare_model_for_kbit_training(model)

In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],   # (Attention-only)
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [ ]:
from trl import DPOConfig

In [ ]:
dpo_config = DPOConfig(
    output_dir="./qwen2.5-1.5b-anti-sycophancy",
    learning_rate=1e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    beta=0.2,
    loss_type="sigmoid_norm",
    fp16=False,
    bf16=False,
    max_length=1024,
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=10,
    report_to="none"
)

In [ ]:
from trl import DPOTrainer

In [ ]:
trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer
)

Tokenizing train dataset:   0%|          | 0/2545 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/2545 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/283 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset:   0%|          | 0/283 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
50,0.692094,0.691911,1.694460,88242.000000,-0.796807,-0.186253,0.521514,-0.006773,-0.123978,0.968750,0.117205,-154.746596,-136.574412
100,0.691028,0.690928,1.691815,176661.000000,-0.807581,-0.200313,0.521911,-0.012088,-0.222543,0.968750,0.210455,-154.773173,-137.067239
150,0.690360,0.690108,1.689606,264356.000000,-0.816671,-0.211960,0.521675,-0.018602,-0.306474,0.968750,0.287872,-154.805742,-137.486894
200,0.689206,0.689349,1.687311,352556.000000,-0.825556,-0.223519,0.521616,-0.026348,-0.385463,0.968750,0.359115,-154.844474,-137.881838
250,0.689103,0.689151,1.686568,441394.000000,-0.828256,-0.226956,0.521253,-0.030408,-0.407586,0.968750,0.377178,-154.864773,-137.992453
300,0.689347,0.689148,1.686599,528417.000000,-0.828238,-0.226941,0.521054,-0.029730,-0.407541,0.968750,0.377811,-154.861386,-137.992231
319,0.689227,0.689142,1.686624,560447.000000,-0.828152,-0.226978,0.521478,-0.029435,-0.407906,0.968750,0.378471,-154.859908,-137.994056


TrainOutput(global_step=319, training_loss=0.6903417715831984, metrics={'train_runtime': 1874.2961, 'train_samples_per_second': 1.358, 'train_steps_per_second': 0.17, 'total_flos': 5503938996092928.0, 'train_loss': 0.6903417715831984, 'epoch': 1.0})

In [ ]:
trainer.model.push_to_hub("kashishgupta/qwen2.5-1.5b-anti-sycophancy-lora")

README.md:   0%|          | 0.00/2.06k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|          | 25.7kB / 4.39MB            

  ...adapter_model.safetensors:   7%|7         |  637kB / 8.75MB            

CommitInfo(commit_url='https://huggingface.co/kashishgupta/qwen2.5-1.5b-anti-sycophancy-lora/commit/db59f730875f0d04be23bce5331a2397b1bcc70e', commit_message='Upload model', commit_description='', oid='db59f730875f0d04be23bce5331a2397b1bcc70e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/kashishgupta/qwen2.5-1.5b-anti-sycophancy-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='kashishgupta/qwen2.5-1.5b-anti-sycophancy-lora'), pr_revision=None, pr_num=None)